In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import tensorflow as tf

sys.path.append(os.path.abspath("../"))

import numpy as np
import scipy

from train.models import FullModel
from util import dataset as u_dataset
from util import image as u_image
from util import metrics as u_metrics

#### Load Test Data


In [ ]:
# test_directory = "../../data/groundtruth/GORE23_Gott_Gott_CompetitionWalk_Default__HTWK-Robots_1stHalf_1"


# label = u_dataset.load_labels(test_directory)[0]

# camera = u_dataset.camera_from_label(label)
# camera = tf.constant(camera, dtype=tf.float32)

# intrinsics = u_dataset.intrinsics_from_label(label)
# intrinsics = tf.constant(intrinsics, dtype=tf.float32)

# image = u_dataset.load_image(test_directory, label, image_format=u_image.ImageFormat.YUYV)
# image_yuv = tf.reshape(tf.constant(image), (-1, 480, 320, 4))

In [ ]:
test_directory = "../../data/prelabeled"
groundtruth_directory = "../../data/groundtruth"

test_data_info = u_dataset.get_data_info(test_directory)
groundtruth_data_info = u_dataset.get_data_info(groundtruth_directory)

test_dataset = u_dataset.get_dataset(test_data_info["file_names"])
groundtruth_dataset = u_dataset.get_dataset(groundtruth_data_info["file_names"])

print("Number of test samples: ", test_data_info["num_samples"])
print("Number of groundtruth samples: ", groundtruth_data_info["num_samples"])

In [ ]:
# masks = u_dataset.get_masks(label, "ball")
# coords = u_dataset.get_coords_from_offsets(masks[0])
# # print(masks[0])
# print(coords)

In [ ]:
a = np.array([[0.1, 0.0, 0.0], [0.0, 0.0, 0.0]])

print(a == 0.0)
print((a == 0).all())

#### Accuracy, Precision, Recall


In [ ]:
# precision_metric = tf.keras.metrics.Precision()
# recall_metric = tf.keras.metrics.Recall()
# accuracy_metric = tf.keras.metrics.BinaryAccuracy()

# mae_metric = tf.keras.metrics.MeanAbsoluteError()
mae_metric = u_metrics.MAE()
cnt = 0

for test_data, groundtruth_data in zip(test_dataset, groundtruth_dataset, strict=True):
    cnt += 1
    # precision_metric.update_state(groundtruth_data["object_ball"], test_data["object_ball"])
    # accuracy_metric.update_state(groundtruth_data["object_ball"], test_data["object_ball"])
    mae_metric.update_state(groundtruth_data, test_data)

    # if cnt == 10:
    #     break


# precision = precision_metric.result().numpy()
# accuracy = accuracy_metric.result().numpy()
mae = mae_metric.result().numpy()

print(mae)

In [ ]:
def match_keypoints(kps, pts, e_max=20):
    """Matches predicted and labeled points optimally. Points are only matched as
    long as their distance is below or equal to e_max. It does not match two
    points from kps to the same point in pts or vice versa. Correspondingly it can
    happen that some points remain unmatched.

    A matrix is constructed that contains a score for each pair of detected and
    labeled point. The score is 0 for pairs that are at least e_max pixels apart
    and 1 for points with distance 0 (cf. the error metric for point detectors from
    lecture 11a).

    Then scipy.optimize.linear_sum_assignment calculates the assignment that maximizes
    the overall score. Dummy assignments (with score 0) are filtered out before the
    matched points are stacked together to obtain the result.

    :param kps: numpy array of predicted keypoints (Mx2)
    :param pts: numpy array of labeled points (Nx2)
    :param e_max: max pixel distance to consider a match
    :return: numpy array of matches (k, p), where k is from kps and p the matched point from pts (Kx2x2)
    """
    if kps.size == 0 or pts.size == 0:
        return np.zeros((0, 2, 2), dtype=kps.dtype)
    # TODO: Implement (4P)
    # MUSTERLÖSUNG
    diffs = kps[:, np.newaxis] - pts[np.newaxis]
    score_matrix = np.maximum(1 - np.linalg.norm(diffs, axis=-1) / e_max, 0)
    print("Score matrix: \n", score_matrix)
    row_ind, col_ind = scipy.optimize.linear_sum_assignment(score_matrix, maximize=True)
    print("Row ind: ", row_ind)
    print("Col ind: ", col_ind)
    assigned = score_matrix[row_ind, col_ind] > 0

    print("Assigned: ", assigned)

    return np.stack([kps[row_ind[assigned]], pts[col_ind[assigned]]], axis=1)

In [ ]:
wrld_coords_pred = tf.constant(
    [
        [3.6778605, -0.47758237, -0.4814842],
        [360.6778605, -60.47758237, -20.4814842],
        [37.6778605, -4.47758237, -2.4814842],
        [0.6778605, -6.47758237, -50.4814842],
    ],
    shape=(4, 3),
    dtype=tf.float32,
).numpy()
wrld_coords_true = tf.constant(
    [
        [3.8324456, -0.50522536, -0.4814842],
        [37.8324456, -4.50522536, -2.4814842],
        [360.8324456, -60.50522536, -20.4814842],
    ],
    shape=(3, 3),
    dtype=tf.float32,
).numpy()

print(wrld_coords_pred)
print(wrld_coords_true)


match = match_keypoints(kps=wrld_coords_pred, pts=wrld_coords_true, e_max=0.20)

print("Match: \n", match)


for pair in match:
    print("Dist: ", np.linalg.norm(pair[1] - pair[0]))

#### Load Encoder Model


In [ ]:
# Compile Full Model
model = FullModel(480, 320)
model.compile(optimizer=tf.keras.optimizers.Adam())

model.encoder.load_weights("../../models/encoder/encoder_overfit.keras")

# Get Patch Extractor Layer from Full Model
patch_sampler = model.get_layer("patch_sampler")
patch_extractor = model.get_layer("patch_extractor")

#### Show Raw Image


In [ ]:
plt.imshow(image[..., 0], cmap="gray")
plt.title("Raw Example Image in Grayscale")
plt.show()

#### Show Groundtruth Masks on Image


In [ ]:
u_image.show_masks_on_image(test_directory, label, "ball", mask_name=None, grid_dims=(15, 20))
u_image.show_masks_on_image(
    test_directory, label, "ball", mask_name="objectness", grid_dims=(15, 20)
)
u_image.show_masks_on_image(test_directory, label, "ball", mask_name="loss", grid_dims=(15, 20))

#### Show Predicted Patches


In [ ]:
results = model((image_yuv, camera, intrinsics), training=None)

print(results)

u_image.show_patches_on_image(image, "ball", results)